# ДЗ-1. Три вычислительные формы causal linear operator

Курс «Дополнительные архитектуры генеративного ИИ», ФКН НИУ ВШЭ, программа ИПИИ.

Выдаётся 15.09, сдаётся 22.09. Вес 30 % итоговой оценки.
Трудоёмкость 8-10 часов. Все вычисления на CPU.

## Нотация

Позиции $t=1,\ldots,T$. Канонический порядок осей `[B, H, T, D]`.

$$
q_t,k_t\in\mathbb R^{d_k},\qquad v_t\in\mathbb R^{d_v}
$$

Равенство $d_k=d_v$ **не предполагается**.

Состояние $S_t\in\mathbb R^{d_v\times d_k}$, по умолчанию $S_0=0$.
С батчем и головами состояние имеет форму `[B, H, d_v, d_k]`.

Запись и чтение:

$$
S_t=S_{t-1}+v_tk_t^\top,\qquad y_t=S_tq_t
$$

Порядок операций внутри шага фиксирован: **сначала** в состояние записывается
текущий $v_tk_t^\top$, **затем** вычисляется $y_t$.

Маска inclusive:

$$
M_{ti}=\mathbf 1[i\leq t]
$$

Определение оператора, от которого всё остальное является производным:

$$
y_t=S_0q_t+\sum_{i=1}^{t}(q_t^\top k_i)\,v_i
$$

## Три вычислительные формы

Все три вычисляют **один и тот же** оператор. Различаются они алгоритмом, а не
результатом, поэтому по ответу отличить их невозможно - отличаются они тем, что
именно происходит при вычислении.

**Parallel.** Строится матрица взаимодействий $QK^\top$ размера $T\times T$,
применяется маска, результат умножается на $V$:

$$
Y=\left(\left(QK^\top\right)\odot M\right)V+QS_0
$$

Ответ вычисляется **через** эту матрицу: она не декоративна.

**Recurrent.** Последовательность обходится один раз. Состояние обновляется
**ровно один раз на позицию**, внутриблочные матрицы не строятся:

$$
S_t=S_{t-1}+v_tk_t^\top,\qquad y_t=S_tq_t
$$

**Chunkwise.** Последовательность режется на блоки заданного размера. Внутри
блока используется локальная матрица взаимодействий, между блоками переносится
состояние:

$$
Y_b=Q_b(S_b^{\rm in})^\top+\left(\left(Q_bK_b^\top\right)\odot M_b\right)V_b,
\qquad
S_b^{\rm out}=S_b^{\rm in}+V_b^\top K_b
$$

Число операций в chunkwise зависит от размера блока - это и отличает её от двух
остальных.

## Контракт функций

Имена и порядок аргументов - те же, что на семинаре 1: `la_parallel`,
`la_recurrent`, `la_chunkwise(q, k, v, C=64)`. Свои функции с семинара можно
перенести сюда как есть; здесь к ним добавляются две вещи, которых на семинаре
не было.

**Начальное состояние `state0`.** Необязательный последний аргумент. Если он
передан, оператор стартует не с нуля, а с него.

**Второй возврат `final_state`.** Каждая функция возвращает пару, а не один
тензор: состояние после всех $T$ записей нужно и само по себе, и как вход для
следующего блока.

Входы:

- `q`: обычный strided CPU tensor `[B, H, T, d_k]` с плавающей точкой;
- `k`: обычный strided CPU tensor `[B, H, T, d_k]`;
- `v`: обычный strided CPU tensor `[B, H, T, d_v]`;
- `state0`: `None` либо обычный strided CPU tensor `[B, H, d_v, d_k]`;
- `C`: только для `la_chunkwise`, положительный `int`, по умолчанию 64.

Предусловия: $B,H,T,d_k,d_v\geq1$. Все входы имеют одинаковые dtype и device.
Обязательные dtype - `torch.float32` и `torch.float64`. Sparse-тензоров не
будет. Contiguous не требуется: допустимы транспонирование, срез с шагом больше
единицы и ненулевой `storage_offset`, в любом сочетании.

**Численный домен.** Проверочные входы - случайные нормальные величины
умеренного масштаба, порядка единицы. `NaN` и $\pm\infty$ во входах не
встречаются, переполнения ни в одной промежуточной величине трёх форм не
происходит, и суммы не содержат специально подобранного взаимного уничтожения
больших слагаемых.

Оговорка нужна вот почему. Формы суммируют в разном порядке, и на входах вроде
$k = (10^{16},\,1,\,-10^{16},\,1)$ это перестаёт быть погрешностью округления:
$10^{16}+1$ во float64 равно $10^{16}$, единица теряется целиком, и разные
порядки суммирования дают разные ответы - не на $10^{-10}$, а на единицу. На
таких входах эквивалентность форм - утверждение о вещественных числах, а не о
машинных, и проверять её бессмысленно. Входы вне контракта не оцениваются,
исключение для них не задаётся.

Каждая функция возвращает пару `(y, final_state)`:

- `y` имеет форму `[B, H, T, d_v]`;
- `final_state` имеет форму `[B, H, d_v, d_k]` и равен $S_T$;
- обе величины имеют dtype и device, совпадающие с `q`.

Функции не изменяют входы на месте.

**Градиенты.** Для входов с `requires_grad=True` вычисление сохраняет
autograd-граф, и производные верны - не только существуют. `y`
дифференцируется по `q`, `k`, `v` и по `state0`, если он передан. `final_state`
дифференцируется по `k`, `v` и по `state0`; от `q` он математически не зависит.

**Среда.** Решение исполняется в обычном eager-режиме, без `torch.compile` и
`vmap`.

Отдельно, как правила чтения кода, а не автопроверки: не отрывайте граф
(`.detach()`, `torch.no_grad()`, `torch.inference_mode()`), не заводите
глобальных кэшей и не считайте ничего на уровне импорта. Автотесты этого не
видят, преподаватель видит.

**Самостоятельность форм.** Каждая форма вычисляется своим способом.
Параллельная строит матрицу взаимодействий $T\times T$ одним тензором и
использует её в ответе - не разбивая на части и не создавая «для вида».

Разрешено: общий помощник, который создаёт начальное состояние, строит маску,
проверяет формы входов. Такой помощник не содержит вычисления оператора.

Запрещено: помощник, который принимает `q`, `k`, `v` и возвращает готовый
причинный результат, вызываемый из двух форм. Это вынесенный основной алгоритм,
и тогда форм фактически не три, а одна.

**Равенство форм понимается алгебраически.** Порядок суммирования у трёх форм
разный, поэтому в плавающей точке ответы отличаются на величину округления.
Сравнение везде идёт с допуском, побитового совпадения не требуется. Допуск
рассчитан на численный домен, описанный выше.

**Краевые случаи, которые обязаны работать:** $T=1$ и длинные
последовательности; `C` равный $1$, равный $T$ и больший $T$; неполный
последний блок; $d_k\neq d_v$, а также $d_k=1$ и $d_v=1$; ненулевой `state0`;
неконтигуальные входы и неконтигуальный `state0`.

## Подготовка

Ячейку ниже выполните до начала работы.

In [ ]:
import time
from typing import Optional, Tuple

import torch

torch.manual_seed(20260915)
torch.set_num_threads(1)


def make_inputs(T=8, d_k=3, d_v=5, B=2, H=2, seed=0, dtype=torch.float64):
    """Воспроизводимые входы в контрактных формах."""
    g = torch.Generator().manual_seed(seed)
    mk = lambda *s: torch.randn(*s, generator=g, dtype=dtype)
    return mk(B, H, T, d_k), mk(B, H, T, d_k), mk(B, H, T, d_v)

## Независимый эталон

Эта функция вычисляет определение оператора напрямую, скалярным суммированием,
в FP64. Она медленная, и в этом смысл: с ней сравниваются все три ваши
реализации.

**Совпадение трёх ваших форм между собой ничего не доказывает** - три
одинаково неверные реализации совпадут друг с другом. Сравнивать нужно с
эталоном.

In [ ]:
def oracle_fp64(q, k, v, state0=None):
    """Определение оператора, посчитанное прямыми скалярными циклами в FP64.

    Матричных произведений, масок и рекуррентности здесь нет: это независимый
    эталон, с которым сравниваются все три формы.
    """
    q = q.detach().to(torch.float64)
    k = k.detach().to(torch.float64)
    v = v.detach().to(torch.float64)
    B, H, T, d_k = q.shape
    d_v = v.shape[-1]
    s0 = (torch.zeros(B, H, d_v, d_k, dtype=torch.float64)
          if state0 is None else state0.detach().to(torch.float64))

    y = torch.zeros(B, H, T, d_v, dtype=torch.float64)
    for b in range(B):
        for h in range(H):
            for t in range(T):
                acc = torch.zeros(d_v, dtype=torch.float64)
                for p in range(d_k):
                    acc += q[b, h, t, p] * s0[b, h, :, p]
                for i in range(t + 1):
                    dot = torch.zeros((), dtype=torch.float64)
                    for p in range(d_k):
                        dot += q[b, h, t, p] * k[b, h, i, p]
                    acc += dot * v[b, h, i, :]
                y[b, h, t, :] = acc

    final = s0.clone()
    for b in range(B):
        for h in range(H):
            for i in range(T):
                final[b, h] += torch.outer(v[b, h, i], k[b, h, i])
    return y, final

## Разобранный пример

Возьмём $B=H=1$, $T=2$, $d_k=d_v=1$, нулевое начальное состояние:

$$
q=\begin{pmatrix}2\\3\end{pmatrix},\qquad
k=\begin{pmatrix}1\\4\end{pmatrix},\qquad
v=\begin{pmatrix}5\\6\end{pmatrix}
$$

По определению оператора:

$$
y_1=(q_1k_1)v_1=(2\cdot1)\cdot5=10
$$

$$
y_2=(q_2k_1)v_1+(q_2k_2)v_2=(3\cdot1)\cdot5+(3\cdot4)\cdot6=15+72=87
$$

Состояние после обхода:

$$
S_2=v_1k_1+v_2k_2=1\cdot5+4\cdot6=29
$$

Ячейка ниже сверяет эти числа с эталоном. Она уже работает - на ней удобно
убедиться, что вы правильно поняли порядок осей.

In [ ]:
q_ex = torch.tensor([[[[2.0], [3.0]]]], dtype=torch.float64)
k_ex = torch.tensor([[[[1.0], [4.0]]]], dtype=torch.float64)
v_ex = torch.tensor([[[[5.0], [6.0]]]], dtype=torch.float64)

y_hand = [10.0, 87.0]
s_hand = [29.0]

y_ref, s_ref = oracle_fp64(q_ex, k_ex, v_ex)

print("формы входов:", tuple(q_ex.shape), tuple(k_ex.shape), tuple(v_ex.shape))
print("посчитано руками:  y =", y_hand, " S_T =", s_hand)
print("эталон вернул:     y =", y_ref.flatten().tolist(),
      " S_T =", s_ref.flatten().tolist())
assert y_ref.flatten().tolist() == y_hand
assert s_ref.flatten().tolist() == s_hand
print("\nсовпало")

## Задание 1. Parallel form

Реализуйте параллельную форму. Ответ должен вычисляться **через** матрицу
взаимодействий $T\times T$, а не мимо неё.

In [ ]:
def la_parallel(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    state0: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Параллельная форма causal linear operator.

    Возвращает
    ----------
    y : [B, H, T, d_v]
    final_state : [B, H, d_v, d_k], равен S_T
    """
    raise NotImplementedError

## Задание 2. Recurrent form

Реализуйте рекуррентную форму. Состояние обновляется ровно один раз на позицию,
внутриблочные матрицы не строятся, весь префикс заново не пересчитывается.

In [ ]:
def la_recurrent(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    state0: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Рекуррентная форма causal linear operator.

    Возвращает
    ----------
    y : [B, H, T, d_v]
    final_state : [B, H, d_v, d_k], равен S_T
    """
    raise NotImplementedError

## Задание 3. Chunkwise form

Реализуйте блочную форму. Число операций обязано зависеть от `C`.

In [ ]:
def la_chunkwise(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    C: int = 64,
    state0: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Блочная форма causal linear operator.

    Возвращает
    ----------
    y : [B, H, T, d_v]
    final_state : [B, H, d_v, d_k], равен S_T
    """
    raise NotImplementedError

## Самопроверка

Здесь несколько базовых проверок. Полный комплект применяется при проверке
работы и шире этого набора: причинность, градиенты обеих возвращаемых величин,
поведение при `state0 != 0`, краевые размерности, неконтигуальные входы, обе
точности.

Соответствие вычислительной форме автотесты не проверяют - это ручная часть,
см. «Как оценивается».

Проходящая самопроверка означает, что вы движетесь верно, а не что работа
закончена.

**Всё, что ниже этого заголовка, при проверке не исполняется.** Оцениваемый код
- три реализации и то, на что они опираются, - должен быть выше.

In [ ]:
def _close(got, want, rtol=1e-10, atol=1e-12):
    if len(got) != len(want):
        return False
    return all(torch.allclose(g, w, rtol=rtol, atol=atol)
               for g, w in zip(got, want))


def _try(thunk):
    """Возвращает True, False или None, если функция ещё не реализована."""
    try:
        return bool(thunk())
    except NotImplementedError:
        return None
    except Exception:
        return False


def selfcheck():
    cases = []

    q, k, v = make_inputs(T=8, d_k=3, d_v=5)
    want_y, want_s = oracle_fp64(q, k, v)
    for name, fn, args in (
        ("parallel", la_parallel, (q, k, v)),
        ("recurrent", la_recurrent, (q, k, v)),
        ("chunkwise", la_chunkwise, (q, k, v, 3)),
    ):
        cases.append((f"{name}: совпадает с эталоном",
                      _try(lambda: _close(fn(*args), (want_y, want_s)))))

    q1, k1, v1 = make_inputs(T=1, d_k=2, d_v=3)
    cases.append(("T=1: маска включает диагональ",
                  _try(lambda: _close(la_parallel(q1, k1, v1)[:1],
                                      oracle_fp64(q1, k1, v1)[:1]))))

    q2, k2, v2 = make_inputs(T=7, d_k=4, d_v=2)
    cases.append(("неполный последний блок",
                  _try(lambda: _close(la_chunkwise(q2, k2, v2, 3)[:1],
                                      oracle_fp64(q2, k2, v2)[:1]))))

    B, H, T, d_k, d_v = 2, 2, 6, 4, 2
    s0 = torch.randn(B, H, d_v, d_k, dtype=torch.float64)
    q3, k3, v3 = make_inputs(T=T, d_k=d_k, d_v=d_v, B=B, H=H, seed=5)
    cases.append(("ненулевое начальное состояние",
                  _try(lambda: _close(la_recurrent(q3, k3, v3, s0)[:1],
                                      oracle_fp64(q3, k3, v3, s0)[:1]))))

    q4, k4, v4 = make_inputs(T=5, d_k=6, d_v=2)
    cases.append(("d_k != d_v: форма выхода",
                  _try(lambda: tuple(la_parallel(q4, k4, v4)[0].shape)
                       == (2, 2, 5, 2))))

    mark = {True: "OK     ", False: "FAIL   ", None: "не готов"}
    for name, ok in cases:
        print(f"{mark[ok]}  {name}")
    done = sum(1 for _, o in cases if o is True)
    todo = sum(1 for _, o in cases if o is None)
    print(f"\nпройдено {done} из {len(cases)}"
          + (f", ещё не реализовано: {todo}" if todo else ""))


selfcheck()

## Измерения

Измеряются **время** и **пиковая память** трёх форм. Метрики и протокол заданы
ниже полностью: два аккуратных студента должны получить сопоставимые между собой
числа на одной машине.

### Что считается памятью

Готового счётчика памяти для CPU-тензоров в PyTorch нет, поэтому метрика задана
и функция замера дана готовой: **пик логических байтов среди тензоров, которые
создал и увидел диспетчер внутри замеряемого вызова** - сумма
`numel * element_size` по живым в этот момент тензорам, взятая в максимуме по
ходу вызова. Не сумма выделений, а одновременный объём.

Это прокси, а не настоящий пик занятой памяти, и отношения «не больше» или «не
меньше» к реальному потреблению у неё нет:

- в одну сторону завышает - view вроде транспонирования, среза или `unsqueeze`
  новых данных не создаёт, но считается полным объёмом;
- в другую занижает - входы `q`, `k`, `v`, `state0` не учитываются, как и
  рабочие буферы библиотек, накладные расходы аллокатора и объекты Python.

Для сравнения трёх форм между собой она годится: считается для всех одинаково.
В анализе на это стоит сослаться.

### Сетка запусков

- $T \in \{64, 128, 256, 512, 1024\}$ при фиксированных $B=1$, $H=1$,
  $d_k=32$, $d_v=32$, `C = 64`;
- отдельно `C` $\in \{1, 8, 32, 128, T\}$ при фиксированном $T=512$ -
  только для chunkwise;
- `dtype` = `torch.float32`, один поток;
- входы получаются вызовом `make_inputs` с параметрами замера и **`seed=0`**;
  `state0` не передаётся;
- на одну длину $T$ входы создаются один раз и переиспользуются всеми тремя
  формами; в развёртке по `C` тоже один набор входов на все пять
  размеров блока;
- порядок обхода: сначала вся сетка по $T$, потом вся сетка по `C`;
- перед каждым замером **2 прогревочных вызова**, затем **7 повторов**;
- время: медиана семи повторов; **разброс** = разность максимума и минимума;
- память измеряется одним вызовом, повторы не нужны: величина детерминирована.

### Схема `measurements.csv`

Драйвер замеров дан готовым ниже: протокол одинаковый у всех, и писать его
заново не нужно. Ваша часть здесь - разбор полученных чисел.

Колонки ровно эти, в этом порядке, одна строка на измерение:

`form,T,chunk_size,B,H,d_k,d_v,dtype,threads,repeats,median_ms,spread_ms,peak_bytes`

Допустимые значения: `form` - `parallel`, `recurrent`, `chunkwise`; `dtype` -
`float32`; колонка `chunk_size` - это тот же `C`, для parallel и recurrent она
пустая; `threads` - 1.
Время в миллисекундах, дробное; `peak_bytes` - целое.

Первые строки должны выглядеть так:

```
form,T,chunk_size,B,H,d_k,d_v,dtype,threads,repeats,median_ms,spread_ms,peak_bytes
parallel,64,,1,1,32,32,float32,1,7,0.048,0.01,81920
recurrent,64,,1,1,32,32,float32,1,7,1.269,0.155,28672
chunkwise,64,64,1,1,32,32,float32,1,7,0.057,0.008,98304
```

Строк должно получиться $3\cdot5 + 5 = 20$: пять длин на три формы плюс пять
размеров блока для chunkwise.

In [ ]:
import weakref

from torch.utils._python_dispatch import TorchDispatchMode


class PeakMemory(TorchDispatchMode):
    """Пик одновременно живущих байтов среди тензоров, созданных внутри блока.

    На каждый созданный тензор вешается weakref: освобождение уменьшает счётчик,
    поэтому считается именно пик, а не сумма выделений.
    """

    def __init__(self):
        self.live = 0
        self.peak = 0
        self._refs = []

    def __torch_dispatch__(self, func, types, args=(), kwargs=None):
        out = func(*args, **(kwargs or {}))
        for t in torch.utils._pytree.tree_flatten(out)[0]:
            if isinstance(t, torch.Tensor):
                nbytes = t.numel() * t.element_size()
                self.live += nbytes
                self.peak = max(self.peak, self.live)

                def _release(_ref, n=nbytes, rec=self):
                    rec.live -= n

                self._refs.append(weakref.ref(t, _release))
        return out


def measure_peak_bytes(fn, *args, **kwargs):
    """Пик живых байтов при одном вызове fn. Величина детерминирована."""
    meter = PeakMemory()
    with meter:
        fn(*args, **kwargs)
    return meter.peak

In [ ]:
GRID = {
    'lengths': (64, 128, 256, 512, 1024),
    'chunk_sizes': (1, 8, 32, 128, 512),
    'chunk_sweep_T': 512,
    'chunk_size': 64,
    'B': 1,
    'H': 1,
    'd_k': 32,
    'd_v': 32,
    'threads': 1,
    'repeats': 7,
    'warmup': 2,
    'seed': 0,
}


def _time_call(fn, args, repeats, warmup):
    """Медиана и разброс в миллисекундах."""
    for _ in range(warmup):
        fn(*args)
    samples = []
    for _ in range(repeats):
        started = time.perf_counter()
        fn(*args)
        samples.append((time.perf_counter() - started) * 1000)
    samples.sort()
    return samples[len(samples) // 2], samples[-1] - samples[0]


def run_measurements(path="measurements.csv", cfg=GRID):
    """Прогон по сетке из условия и запись CSV. Драйвер дан готовым.

    Ваша часть - три реализации выше и разбор полученных чисел ниже. Здесь
    менять ничего не нужно: протокол задан условием, и одинаковый драйвер у
    всех делает замеры сравнимыми.
    """
    torch.set_num_threads(cfg["threads"])
    B, H = cfg["B"], cfg["H"]
    d_k, d_v = cfg["d_k"], cfg["d_v"]
    rows = []

    def add(form, T, chunk, fn, args):
        median, spread = _time_call(fn, args, cfg["repeats"], cfg["warmup"])
        rows.append([form, T, chunk, B, H, d_k, d_v, "float32",
                     cfg["threads"], cfg["repeats"],
                     round(median, 3), round(spread, 3),
                     measure_peak_bytes(fn, *args)])

    for T in cfg["lengths"]:
        q, k, v = make_inputs(T=T, d_k=d_k, d_v=d_v, B=B, H=H,
                              seed=cfg["seed"], dtype=torch.float32)
        add("parallel", T, "", la_parallel, (q, k, v))
        add("recurrent", T, "", la_recurrent, (q, k, v))
        add("chunkwise", T, cfg["chunk_size"], la_chunkwise,
            (q, k, v, cfg["chunk_size"]))

    T = cfg["chunk_sweep_T"]
    q, k, v = make_inputs(T=T, d_k=d_k, d_v=d_v, B=B, H=H,
                          seed=cfg["seed"], dtype=torch.float32)
    for chunk in cfg["chunk_sizes"]:
        add("chunkwise", T, chunk, la_chunkwise, (q, k, v, chunk))

    header = ("form,T,chunk_size,B,H,d_k,d_v,dtype,threads,"
              "repeats,median_ms,spread_ms,peak_bytes")
    with open(path, "w", encoding="utf-8") as f:
        f.write(header + "\n")
        for row in rows:
            f.write(",".join(str(x) for x in row) + "\n")
    print(f"записано строк: {len(rows)} -> {path}")
    return rows


# раскомментируйте, когда три формы готовы: прогон занимает около минуты
# run_measurements()

## Письменный вывод

Ответы пишите прямо в ячейках ниже, в markdown.

### 1. Вывод chunkwise-формы

Выведите блочную форму из определения оператора. Обязательно явно введите
отображение локального индекса строки блока в глобальный индекс токена и
покажите, где оно используется.

*Ваш ответ:*

### 2. Почему три формы задают один оператор

Обоснуйте, что три формы вычисляют одну и ту же функцию, а не приближают друг
друга. Объясните, почему их взаимное совпадение не является доказательством
корректности.

*Ваш ответ:*

### 3. Сложность обучения и декодирования

Два режима заданы так:

- **обучение** - вычисление всей известной последовательности длины $T$ за один
  вызов;
- **декодирование** - получение одного нового токена, когда состояние
  $S_{t-1}$ уже сохранено.

Память считайте так: всё, что живёт одновременно во время вызова при
`requires_grad=False`, **кроме входов**. Возвращаемые `y` и `final_state`
считаются: они обязаны существовать у любой формы. Тогда у всех трёх есть общее
слагаемое $O(Td_v + d_kd_v)$, а различает формы то, что сверх него.

Для каждой из трёх форм укажите сложность обучения по времени и по памяти -
через $T$, $d_k$, $d_v$ и размер блока $c$, а не через одно общее $d$. Общее
слагаемое можно назвать один раз и дальше писать только различающую часть. Затем
разберите декодирование отдельно: какие из трёх форм в этом режиме остаются
самостоятельным способом вычисления, а какие сводятся к одной и той же операции.
Укажите, где возникает объект размера $T\times T$ и где его нет.

*Ваш ответ:*

## Анализ измерений

Опишите, что показали ваши замеры: как растёт время и память по длине
последовательности, как ведёт себя chunkwise при изменении размера блока,
совпадает ли наблюдаемое с ожидаемой асимптотикой и где расходится.

Отдельно укажите, какая часть измеренного относится к реализации, а не к форме.

*Ваш ответ:*

## Использование генеративных моделей

Использование разрешено для поиска идей, навигации по литературе, чернового
кода, отладки и языкового редактирования.

Заполните раздел ниже: какие модели использовались, для каких задач, какие
существенные части работы получены или изменены с их помощью. Существенное
использование без описания рассматривается как нарушение правил выполнения
работы.

Ответственность за весь сданный материал лежит на авторе независимо от способа
получения. Ошибка генеративной модели оценивается как ошибка в работе.

*Ваш ответ:*

## Как оценивается

| Критерий | Вес | Что оценивается |
|---|---|---|
| Корректность | 20 % | три формы согласованы между собой и с контрактом, работа воспроизводится в чистом ядре |
| Письменный вывод | 30 % | выкладки приведены полностью, каждый переход обоснован, оценка сложности объяснена |
| Измерения | 25 % | замеры воспроизводятся одним прогоном, условия зафиксированы и описаны, сделаны повторы |
| Анализ и выводы | 25 % | наблюдения проинтерпретированы, расхождение расчёта с измерением объяснено, ограничения названы |

Двадцать процентов за автотесты - сознательно небольшая доля. Основной вес
приходится на вывод, качество измерений и анализ, то есть на то, чего нельзя
получить запуском чужого кода.

В корректность входит и то, чего автотесты не видят целиком: **каждая форма
должна быть собой**. Ответ у всех трёх один, и по значению их не различить,
поэтому соответствие определению проверяется чтением кода по разделу «Три
вычислительные формы».

Что читается в коде:

- **parallel** - сам причинный результат получается без обхода позиций по
  одной; матрица взаимодействий размера $T\times T$ строится **одним тензором**
  и участвует в ответе. Это проверяется автотестом: тензор портится в момент
  создания, и ответ обязан испортиться вместе с ним. Цикл для построения маски
  допустим: он не вычисляет ответ;
- **recurrent** - позиции обходятся по одной, и состояние переиспользуется, а не
  накапливается списком. Тензоры, которые autograd сохраняет сам для обратного
  прохода, хранением траектории не считаются;
- **chunkwise** - обход идёт блоками, размер блока влияет на ход вычисления,
  внутри блока работа параллельная, между блоками последовательная.

Реализация, которая считает верно, но своему определению не соответствует,
проходит автотесты и теряет эту строку. Сюда же относятся правила из контракта,
которые автотесты не видят: оторванный граф, глобальные кэши, вычисления на
уровне импорта.

## Как сдавать

Сдаётся **этот же ноутбук**, заполненный, с сохранёнными результатами
выполнения ячеек, и файл `measurements.csv`.

Перед сдачей выполните ноутбук сверху вниз в чистом ядре и убедитесь, что он
проходит целиком.

Оценивается код **выше раздела «Самопроверка»**: три реализации и то, на что они
опираются. Всё ниже при проверке не исполняется. Магии IPython (`%`, `%%`) в
оцениваемой части быть не должно.

На прогон комплекта на вашей работе отводится 120 секунд.

Просрочка: до 24 часов −20 %, от 24 до 48 часов −40 %, от 48 до 72 часов −60 %.
Позже 72 часов работа не принимается. Отсчёт от 23:59 22.09 по времени Москвы.